# DS, Mini Project : ESS 배터리 수명 예측 (DAY1)

박준형(U015)

Batch 1·2·3의 수명 분포, 방전 용량, ΔQ(V), 충전 조건, 초기 신호를 비교했습니다. 실제 추출·피처 계산 코드는 `src/day1_eda.py`, 비교 코드는 `src/day1_compare.py`에 있다. 이 노트북은 **실행 결과를 확인하는 읽기용 노트북**이며, 그래프 해석과 결정 근거는 `DAY1-REPORT.md`에 기록했다. 원본 `.mat` 파일은 `data/`에 둔다.

원논문의 Batch 2는 `2017-06-30` 파일입니다. Batch 1의 미종료·연속 측정 셀과 Batch 3 노이즈 채널을 공개 코드 기준으로 처리한 41·43·40셀을 사용합니다.


## 재생성

아래 셀을 실행하면 각 배치가 별도 프로세스에서 처리된다. RAM 16GB 환경에서도 배치 사이에 메모리를 비울 수 있도록 했다. 시간이 걸리므로 이미 `results/`가 있으면 건너뛰어도 된다.

In [ ]:
from pathlib import Path
import subprocess, sys
ROOT = Path.cwd()
if not (ROOT / 'results/batch_summary.csv').exists():
    for batch in ('batch1','batch2','batch3'):
        subprocess.run([sys.executable, 'src/day1_eda.py', batch], check=True, cwd=ROOT)
    subprocess.run([sys.executable, 'src/day1_compare.py'], check=True, cwd=ROOT)
else:
    print('검증된 results/ 파일을 사용합니다. 원본부터 다시 분석하려면 results/batch_summary.csv를 제거하고 실행하세요.')

## 1. 수명 분포와 짧은 수명 셀

배치별 분포와 500사이클 미만·1,000사이클 초과 비율을 비교합니다. 가장 짧은 셀의 충전 정책과 초기 신호도 확인합니다. 수명 레이블이 없는 셀은 비율의 분모에서 제외합니다.

In [ ]:
import pandas as pd
from IPython.display import display, Image
summary = pd.read_csv('results/batch_summary.csv')
display(summary)
display(Image(filename='results/figures/compare_life.png'))
cells = pd.read_csv('results/paper_screened_cells.csv')
display(cells.nsmallest(8, 'cycle_life')[['batch','cell_id','cycle_life','policy','delta_q_logvar']].round(3))

## 2. 방전 용량 곡선과 탐색적 knee

세 배치의 QD 곡선을 함께 비교합니다. 두 직선으로 근사한 knee는 후반 열화를 설명하는 데만 사용하며 예측 피처에 넣지 않습니다.

In [ ]:
display(pd.read_csv('results/knee_summary.csv'))
display(Image(filename='results/figures/compare_qd.png'))

## 3. 초기 ΔQ(V)와 파생변수

`Qdlin`의 100번째와 10번째 사이클 차이를 계산했다. 핵심 후보는 `log10 var(ΔQ(V))`이다. Batch 1에서만 강한 변수가 아니라 세 배치에서 관계 방향이 유지되는지 확인한다.

In [ ]:
display(Image(filename='results/figures/compare_delta_q.png'))
for batch in ('batch1','batch2','batch3'):
    display(Image(filename=f'results/figures/{batch}_delta_q.png'))

## 4. 충전 정책과 열화 속도

첫 구간 C-rate와 충전 시간뿐 아니라, 0~80% SOC 구간의 C-rate를 구간 길이로 가중한 정책 강도를 비교합니다. 실제 평균 전류를 뜻하는 값은 아닙니다.

In [ ]:
policy = pd.read_csv('results/policy_summary.csv')
display(policy.sort_values(['batch','mean_life']))
display(Image(filename='results/figures/compare_c_rate.png'))
display(Image(filename='results/figures/compare_policy_rate.png'))

## 5. 초기 신호의 상관관계와 변수 중복

초기 신호와 수명의 상관을 배치별로 비교합니다. Batch 1에서는 정책 강도와 ΔQ(V)의 상관도 높아 두 변수의 독립적인 기여를 단정하지 않습니다.

In [ ]:
corr = pd.read_csv('results/feature_correlations.csv')
display(corr.pivot(index='feature', columns='batch', values='spearman').round(3))
display(Image(filename='results/figures/compare_charge_time.png'))
display(Image(filename='results/figures/batch1_feature_heatmap.png'))
display(pd.read_csv('results/paper_screened_cells.csv').head())

## 6. 후보 모델 선정과 Batch 1 사전 비교

중앙값 예측을 기준선으로 두고, 변수 중복과 적은 셀 수를 고려해 Ridge·Elastic Net을 비교합니다. 얕은 트리는 비선형 관계를 확인하는 후보입니다. 충전 정책별 5분할 검증에서 ΔQ(V) 단독 Ridge와 핵심 세 변수 Elastic Net이 약 12.97%였습니다. Day 2에는 로그 타깃 Ridge를 포함해 후보를 다시 비교합니다. Batch 2 점수는 이 단계에서 계산하지 않았습니다.

In [ ]:
from pathlib import Path
import subprocess, sys
if not Path('results/day1_probe_summary.csv').exists():
    subprocess.run([sys.executable,'src/day1_model_probe.py'],check=True)
probe = pd.read_csv('results/day1_probe_summary.csv')
display(probe.round(2))
display(Image(filename='results/figures/day1_probe.png'))
pred = pd.read_csv('results/day1_probe_predictions.csv')
ridge = pred[pred.experiment == 'ridge_compact']
display(ridge.sort_values('abs_error',ascending=False).head(8).round(1))